In [27]:
from pathlib import Path

project_data_dir = next(
    (folder / "ml" / "data" for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "ml" / "data").is_dir()),
    None,
)
if project_data_dir is None:
    raise FileNotFoundError("Could not find the project's ml/data directory.")
output_path = project_data_dir / "yelp" / "yelp_150k.csv"
if output_path.exists():
    print(f"Using existing CSV: {output_path.resolve()}")
else:
    from datasets import load_dataset

    ds = load_dataset("Yelp/yelp_review_full", split="train")
    ds = ds.shuffle(seed=42).select(range(min(150_000, len(ds))))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    ds.to_pandas().to_csv(output_path, index=False)
    print(f"Saved {len(ds):,} reviews to {output_path.resolve()}")

Using existing CSV: E:\sentimentiq\ml\data\yelp\yelp_150k.csv


In [29]:
import pandas as pd

y = pd.read_csv(output_path)
print(y.shape, y.columns.tolist())
print("label 0:", y[y["label"] == 0]["text"].iloc[0][:200])
print("label 4:", y[y["label"] == 4]["text"].iloc[0][:200])

(150000, 2) ['label', 'text']
label 0: We were excited to eat here, it is difficult to find. They were closed at 3 p.m. on a Saturday.
label 4: I stalk this truck.  I've been to industrial parks where I pretend to be a tech worker standing in line, strip mall parking lots, and of course the farmer's market.  The bowls are so so absolutely div


In [31]:
import sys, glob
import pandas as pd
sys.path.append("../src")
from text_cleaning import clean_text  # type: ignore[reportMissingImports]

def score_to_label(score):
    if score >= 4:
        return "Positive"
    elif score == 3:
        return "Neutral"
    else:
        return "Negative"

In [32]:
# 1) Food (wahi tarika use karo jo notebook 02 ke Cell 1 mein chalaya tha)
food = pd.read_csv("../data/Reviews.csv", usecols=["Score", "Text"])
food = food.rename(columns={"Score": "stars", "Text": "text"})
food["source"] = "food"

# 2) Yelp (label 0-4 hai, to +1 karke stars)
yelp = pd.read_csv("../data/yelp_150k.csv")
yelp["stars"] = yelp["label"] + 1
yelp = yelp[["stars", "text"]]
yelp["source"] = "yelp"

# 3) Electronics (folder ki saari CSV)
frames = []
for path in glob.glob("../data/electronics/*.csv"):
    tmp = pd.read_csv(path, low_memory=False)
    if "reviews.rating" in tmp.columns and "reviews.text" in tmp.columns:
        tmp = tmp[["reviews.rating", "reviews.text"]]
        tmp.columns = ["stars", "text"]
        frames.append(tmp)
        print("loaded :", path, len(tmp))
    else:
        print("SKIPPED:", path, tmp.columns.tolist()[:10])

elec = pd.concat(frames, ignore_index=True)
elec["source"] = "electronics"

print(len(food), len(yelp), len(elec))

loaded : ../data/electronics\1429_1.csv 34660
loaded : ../data/electronics\Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv 5000
loaded : ../data/electronics\Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv 28332
568454 150000 67992


In [33]:
data = pd.concat([food, yelp, elec], ignore_index=True)
data["stars"] = pd.to_numeric(data["stars"], errors="coerce")
data = data.dropna(subset=["stars", "text"])
data = data[data["stars"].between(1, 5)]
data["stars"] = data["stars"].round().astype(int)
data = data.drop_duplicates(subset="text")
data["label"] = data["stars"].apply(score_to_label)

print(data.groupby(["source", "label"]).size().unstack())

label        Negative  Neutral  Positive
source                                  
electronics      1840     2052     42981
food            57067    29754    306758
yelp            60259    29861     59880


In [34]:
CAP = 20000
parts = []
for src, grp in data.groupby("source"):
    n = min(grp["label"].value_counts().min(), CAP)
    parts.append(grp.groupby("label").sample(n=n, random_state=42))
    print(src, "->", n, "per class")

balanced = pd.concat(parts, ignore_index=True)
print(balanced.groupby(["source", "label"]).size().unstack())

electronics -> 1840 per class
food -> 20000 per class
yelp -> 20000 per class
label        Negative  Neutral  Positive
source                                  
electronics      1840     1840      1840
food            20000    20000     20000
yelp            20000    20000     20000


In [35]:
balanced["clean_text"] = balanced["text"].apply(clean_text)
balanced = balanced[balanced["clean_text"].str.strip() != ""]
print(len(balanced))

125512


In [36]:
from sklearn.model_selection import train_test_split

strat = balanced["source"] + "_" + balanced["label"]
train_df, test_df = train_test_split(
    balanced, test_size=0.2, random_state=42, stratify=strat
)
print(len(train_df), len(test_df))

100409 25103


In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=80000)
X_train = tfidf.fit_transform(train_df["clean_text"])   # sirf train pe fit
X_test = tfidf.transform(test_df["clean_text"])

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, train_df["label"])
print("Training complete")

Training complete


In [39]:
from sklearn.metrics import accuracy_score, classification_report

test_df = test_df.assign(pred=model.predict(X_test))

print("Overall accuracy:", round(accuracy_score(test_df["label"], test_df["pred"]), 4))
print()
for src, g in test_df.groupby("source"):
    print(f"{src:12} accuracy {accuracy_score(g['label'], g['pred']):.4f}   ({len(g)} test reviews)")
print()
print(classification_report(test_df["label"], test_df["pred"]))

Overall accuracy: 0.7319

electronics  accuracy 0.6857   (1104 test reviews)
food         accuracy 0.7250   (12000 test reviews)
yelp         accuracy 0.7431   (11999 test reviews)

              precision    recall  f1-score   support

    Negative       0.76      0.75      0.75      8368
     Neutral       0.64      0.63      0.64      8368
    Positive       0.80      0.81      0.80      8367

    accuracy                           0.73     25103
   macro avg       0.73      0.73      0.73     25103
weighted avg       0.73      0.73      0.73     25103



In [40]:
tests = [
    "The delivery was very late and the food was cold",
    "Absolutely loved it, great taste and fast shipping",
    "It was okay, nothing special",
    "Phone camera is amazing but battery is terrible",
    "Staff was helpful and polite",
    "The room was dirty and the staff was rude",
    "Customer support never replied to my emails",
    "Battery backup is very poor and charging is slow",
]

for t in tests:
    probs = model.predict_proba(tfidf.transform([clean_text(t)]))[0]
    print(f"{model.classes_[probs.argmax()]:9} {probs.max():.0%}  <- {t}")

Negative  63%  <- The delivery was very late and the food was cold
Positive  97%  <- Absolutely loved it, great taste and fast shipping
Neutral   88%  <- It was okay, nothing special
Neutral   55%  <- Phone camera is amazing but battery is terrible
Positive  57%  <- Staff was helpful and polite
Negative  95%  <- The room was dirty and the staff was rude
Negative  80%  <- Customer support never replied to my emails
Negative  93%  <- Battery backup is very poor and charging is slow


In [41]:
import joblib, os

os.makedirs("../models", exist_ok=True)
joblib.dump(tfidf, "../models/tfidf_v2.pkl")
joblib.dump(model, "../models/sentiment_lr_v2.pkl")
print("v2 saved")

v2 saved
